In [17]:
import zipfile
import os
import hashlib


def comprimir_archivo(nombre_original, nombre_zip):
    with zipfile.ZipFile(nombre_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(nombre_original)
    print(f"✅ Archivo comprimido: {nombre_zip}")


def dividir_archivo(nombre_zip, tamano_bloque=50 * 1024 * 1024, prefijo="parte_"):
    with open(nombre_zip, 'rb') as f:
        i = 0
        while True:
            bloque = f.read(tamano_bloque)
            if not bloque:
                break
            nombre_parte = f"models\\test\\{prefijo}{i:02d}"
            with open(nombre_parte, 'wb') as parte:
                parte.write(bloque)
            print(f"🔹 Generado: {nombre_parte}")
            i += 1


def recomponer_y_verificar(nombre_zip_original, prefijo="parte_"):
    partes = sorted([f for f in os.listdir() if f.startswith(prefijo)])
    nombre_reconstruido = "reconstruido.zip"

    with open(nombre_reconstruido, 'wb') as out:
        for parte in partes:
            with open(parte, 'rb') as p:
                out.write(p.read())

    # Verificar con hash SHA256
    def calcular_hash(nombre_archivo):
        sha256 = hashlib.sha256()
        with open(nombre_archivo, 'rb') as f:
            while chunk := f.read(8192):
                sha256.update(chunk)
        return sha256.hexdigest()

    hash_original = calcular_hash(nombre_zip_original)
    hash_reconstruido = calcular_hash(nombre_reconstruido)

    print(f"🔍 Hash original     : {hash_original}")
    print(f"🔍 Hash reconstruido : {hash_reconstruido}")

    if hash_original == hash_reconstruido:
        print("🎉 Verificación exitosa. Sin pérdida de información.")
    else:
        print("⚠️ ¡Los archivos no coinciden!")


In [15]:
nombre_h5 = "models\\Mammalia\\model_M2.h5"
nombre_zip = "models\\test\\modelo_comprimido.zip"
comprimir_archivo(nombre_h5, nombre_zip)


✅ Archivo comprimido: models\test\modelo_comprimido.zip


In [19]:
nombre_zip = "models\\test\\modelo_comprimido.zip"
dividir_archivo(nombre_zip)

🔹 Generado: models\test\parte_00
🔹 Generado: models\test\parte_01
🔹 Generado: models\test\parte_02
🔹 Generado: models\test\parte_03
🔹 Generado: models\test\parte_04


In [5]:
recomponer_y_verificar(nombre_zip)

🔍 Hash original     : 353e5c79e18d45c682ddd122ccc2aeac477c9ef4b775bdd811f37db50f04c40a
🔍 Hash reconstruido : 353e5c79e18d45c682ddd122ccc2aeac477c9ef4b775bdd811f37db50f04c40a
🎉 Verificación exitosa. Sin pérdida de información.


In [ ]:
import zipfile
import os
import hashlib

# -----------------------------
# 🔧 Parámetros editables
# -----------------------------

tamano_bloque = 50 * 1024 * 1024  # ⚠️ Bloques de 50 MB
prefijo = "parte_"

# -----------------------------
# 🗜️ PASO 1: Comprimir el archivo .h5
# -----------------------------
def comprimir_archivo(origen, destino):
    with zipfile.ZipFile(destino, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(origen)
    print(f"✅ Comprimido como {destino}")

# -----------------------------
# 🔪 PASO 2: Dividir en partes pequeñas
# -----------------------------
def dividir_archivo(nombre_zip, tamano_bloque, prefijo):
    with open(nombre_zip, 'rb') as f:
        contador = 0
        while True:
            bloque = f.read(tamano_bloque)
            if not bloque:
                break
            nombre_parte = f"{prefijo}{contador:02d}"
            with open(nombre_parte, 'wb') as parte:
                parte.write(bloque)
            print(f"🔹 Creado: {nombre_parte}")
            contador += 1

# -----------------------------
# 🧩 PASO 3: Reensamblar + verificar
# -----------------------------
def recomponer_y_verificar(nombre_zip_original, prefijo):
    partes = sorted(f for f in os.listdir() if f.startswith(prefijo))
    reconstruido = "reconstruido.zip"

    with open(reconstruido, 'wb') as salida:
        for parte in partes:
            with open(parte, 'rb') as p:
                salida.write(p.read())

    def hash_sha256(nombre):
        sha = hashlib.sha256()
        with open(nombre, 'rb') as f:
            while chunk := f.read(8192):
                sha.update(chunk)
        return sha.hexdigest()

    h_original = hash_sha256(nombre_zip_original)
    h_reconstruido = hash_sha256(reconstruido)

    print(f"🔍 Original   : {h_original}")
    print(f"🔍 Reensamblado: {h_reconstruido}")
    if h_original == h_reconstruido:
        print("🎉 ¡Verificación exitosa!")
    else:
        print("⚠️ Hashes no coinciden. Revisa los pasos.")

# 🧪 Ejecución secuencial
comprimir_archivo(nombre_h5, nombre_zip)
dividir_archivo(nombre_zip, tamano_bloque, prefijo)
recomponer_y_verificar(nombre_zip, prefijo)
